# Starling IDP Ensemble Generation

Generates a conformational ensemble for an intrinsically disordered protein (IDP) region using [Starling](https://github.com/idptools/starling).

**Requires**: GPU runtime (`Runtime > Change runtime type > T4 GPU`)

## What this notebook does
1. Installs Starling + MDAnalysis
2. Runs Starling on a trimmed IDP sequence
3. Extracts individual PDB frames from the XTC trajectory
4. Downloads results as a zip

## Input
- A trimmed IDP sequence (no structured domains — determine boundaries via InterPro first)
- Default example: Tau K18 repeat domain (MTBR, residues 244–368, 2N4R)


## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch to GPU runtime')

## 1. Install Starling

In [ ]:
%%capture
!pip install idptools-starling MDAnalysis

In [ ]:
# Verify install
!starling --help 2>&1 | head -5

## 2. Define your IDP sequence

Paste your trimmed IDP sequence below. **Do not include structured domains** — use InterPro to find the boundary first.

Sequence must be:
- Single-letter amino acid code, no spaces
- Genuinely disordered region only
- Typically 50–300 aa (longer = slower)

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────

PROTEIN_NAME = "tau_k18"          # used for output filenames
N_CONFORMERS = 100                # 100 for exploration, 400 for production

# Tau K18 repeat domain (MTBR, residues 244-368, 2N4R numbering)
# Replace with your sequence — keep it to the IDP region only
SEQUENCE = "KVQIINKKLDLSNVQSKCGSKDNIKHVPGGGSVQIVYKPVDLSKVTSKCGSLGNIHHKPGGGQVEVKSEKLDFKDRVQSKIGSLDNITHVPGGGNKKIETHKLTFRENAKAKTDHGAEIVYKSPVVSGDTSPRHLSNVSSTGSIDMVDSPQLATLADEVSASLAKQGL"

# ────────────────────────────────────────────────────────────────────────────

print(f"Protein  : {PROTEIN_NAME}")
print(f"Length   : {len(SEQUENCE)} aa")
print(f"Conformers: {N_CONFORMERS}")
print(f"Sequence : {SEQUENCE[:40]}...")

## 3. Run Starling

Outputs:
- `{PROTEIN_NAME}_STARLING.pdb` — topology file
- `{PROTEIN_NAME}_STARLING.xtc` — trajectory (N_CONFORMERS frames)

In [ ]:
import os
os.makedirs('starling_output', exist_ok=True)
%cd starling_output

In [ ]:
cmd = f"starling {SEQUENCE} --outname {PROTEIN_NAME} -c {N_CONFORMERS} -r"
print(f"Running: {cmd}\n")
!{cmd}

In [ ]:
import os
pdb_file = f"{PROTEIN_NAME}_STARLING.pdb"
xtc_file = f"{PROTEIN_NAME}_STARLING.xtc"

assert os.path.exists(pdb_file), f"PDB not found: {pdb_file}"
assert os.path.exists(xtc_file), f"XTC not found: {xtc_file}"

print(f"PDB : {pdb_file}  ({os.path.getsize(pdb_file)//1024} KB)")
print(f"XTC : {xtc_file}  ({os.path.getsize(xtc_file)//1024} KB)")

## 4. Extract individual PDB frames

In [ ]:
import MDAnalysis as mda
import numpy as np
import os

def extract_frames(topology, trajectory, output_dir, n_frames=50):
    os.makedirs(output_dir, exist_ok=True)
    u = mda.Universe(topology, trajectory)
    total = len(u.trajectory)
    print(f"Trajectory: {total} frames total")

    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    saved = []
    for i, ts_idx in enumerate(indices):
        u.trajectory[ts_idx]
        out = os.path.join(output_dir, f"frame_{i+1:04d}.pdb")
        u.atoms.write(out)
        saved.append(out)

    print(f"Saved {len(saved)} frames to {output_dir}/")
    return saved


frames_dir = f"{PROTEIN_NAME}_frames"
frames = extract_frames(pdb_file, xtc_file, frames_dir, n_frames=50)
print(f"\nFirst frame: {frames[0]}")

## 5. Quick ensemble analysis

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import numpy as np
import matplotlib.pyplot as plt

u = mda.Universe(pdb_file, xtc_file)
ca = u.select_atoms('name CA')

# Rg across trajectory
rg_values = []
for ts in u.trajectory:
    rg_values.append(ca.radius_of_gyration())

rg = np.array(rg_values)
print(f"Rg  mean={rg.mean():.1f} Å   std={rg.std():.1f} Å   min={rg.min():.1f}   max={rg.max():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rg, linewidth=0.8, color='steelblue')
axes[0].set_xlabel('Conformer index')
axes[0].set_ylabel('Radius of gyration (Å)')
axes[0].set_title(f'{PROTEIN_NAME} — Rg across ensemble')
axes[0].axhline(rg.mean(), color='red', linestyle='--', linewidth=1, label=f'mean={rg.mean():.1f} Å')
axes[0].legend()

axes[1].hist(rg, bins=30, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Radius of gyration (Å)')
axes[1].set_ylabel('Count')
axes[1].set_title('Rg distribution')

plt.tight_layout()
plt.savefig(f'{PROTEIN_NAME}_rg_analysis.png', dpi=150)
plt.show()
print('Plot saved.')

In [ ]:
# End-to-end distance (first CA to last CA) — another disorder metric
u.trajectory[0]
ca_atoms = u.select_atoms('name CA')

ete_values = []
for ts in u.trajectory:
    first = ca_atoms.positions[0]
    last  = ca_atoms.positions[-1]
    ete_values.append(np.linalg.norm(last - first))

ete = np.array(ete_values)
print(f"End-to-end dist  mean={ete.mean():.1f} Å   std={ete.std():.1f} Å   min={ete.min():.1f}   max={ete.max():.1f}")

plt.figure(figsize=(6, 4))
plt.hist(ete, bins=30, color='coral', edgecolor='white')
plt.xlabel('End-to-end distance (Å)')
plt.ylabel('Count')
plt.title(f'{PROTEIN_NAME} — End-to-end distance distribution')
plt.tight_layout()
plt.savefig(f'{PROTEIN_NAME}_ete_analysis.png', dpi=150)
plt.show()

## 6. Download results

In [ ]:
import shutil
from google.colab import files

zip_name = f"{PROTEIN_NAME}_starling_results"
shutil.make_archive(zip_name, 'zip', '.', frames_dir)

# Also include topology + trajectory + plots
import zipfile
with zipfile.ZipFile(f"{zip_name}.zip", 'a') as zf:
    zf.write(pdb_file)
    zf.write(xtc_file)
    for png in [f'{PROTEIN_NAME}_rg_analysis.png', f'{PROTEIN_NAME}_ete_analysis.png']:
        if os.path.exists(png):
            zf.write(png)

print(f"Downloading {zip_name}.zip")
files.download(f"{zip_name}.zip")

## Notes

- **Rg** (radius of gyration): measures compactness. A disordered chain should show a wide distribution — large variance = genuinely disordered. A narrow peak at low Rg = collapsed/structured-like.
- **End-to-end distance**: complementary measure of chain extension. IDPs should span a wide range.
- **Next steps**: upload the `_frames/` PDB files to FoldMason for ensemble QC, or use them directly as Starling-based IDP conformers for binder design.
- **For production**: increase `N_CONFORMERS` to 400 and `n_frames` to 100.
